# CEG5306 HW1: Robot Battery Policy Iteration

This notebook models a robot's charging and task decisions as a finite Markov decision process (MDP).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Robot Battery Policy Iteration notebook initialized.")

## 1. MDP Formulation

The state is

$$
s=(L,B),\qquad L\in\{\texttt{charging\_zone},\texttt{task\_zone}\},\qquad B\in\{0,1,2,3,4,5\}.
$$

The state is Markov because the current location and battery level contain all information needed to determine the legal actions, immediate reward, and next state. The initial-state distribution is deterministic:

$$
P\bigl(S_0=(\texttt{charging\_zone},5)\bigr)=1.
$$

All transitions are deterministic. For each legal state-action pair, the transition probability is 1 for exactly one next state and 0 for every other state.

In [ ]:
LOCATIONS = ("charging_zone", "task_zone")
BATTERY_LEVELS = tuple(range(6))
STATES = tuple(
    (location, battery)
    for location in LOCATIONS
    for battery in BATTERY_LEVELS
)
STATE_TO_INDEX = {state: index for index, state in enumerate(STATES)}

INITIAL_STATE = ("charging_zone", 5)
GAMMA = 0.95

RECHARGE = "recharge"
TRAVEL_TO_TASK = "travel_to_task"
PERFORM_TASK = "perform_task"
RETURN_TO_CHARGING = "return_to_charging"
EMERGENCY_RESCUE = "emergency_rescue"

In [ ]:
def legal_actions(state):
    location, battery = state

    if location == "charging_zone":
        actions = []
        if battery < 5:
            actions.append(RECHARGE)
        if battery >= 1:
            actions.append(TRAVEL_TO_TASK)
        return tuple(actions)

    if battery == 0:
        return (EMERGENCY_RESCUE,)

    return (PERFORM_TASK, RETURN_TO_CHARGING)


def transition(state, action):
    location, battery = state
    if action not in legal_actions(state):
        raise ValueError(f"Illegal action {action!r} for state {state!r}")

    if action == RECHARGE:
        return ("charging_zone", battery + 1), -1.0
    if action == TRAVEL_TO_TASK:
        return ("task_zone", battery - 1), -0.5
    if action == PERFORM_TASK:
        return ("task_zone", battery - 1), 5.0
    if action == RETURN_TO_CHARGING:
        return ("charging_zone", battery - 1), -0.5
    return ("charging_zone", 0), -20.0


assert len(STATES) == 12
assert len(set(STATES)) == 12
assert INITIAL_STATE in STATE_TO_INDEX

for state in STATES:
    actions = legal_actions(state)
    assert actions, f"No legal action for {state}"
    for action in actions:
        next_state, reward = transition(state, action)
        assert next_state in STATE_TO_INDEX
        assert np.isfinite(reward)

print(f"MDP validation passed: {len(STATES)} states")
print(f"Initial state: {INITIAL_STATE}")